# CA1 — Demo: Plotting training diagnostics

This notebook loads training logs and particle arrays (if available) and saves publication-quality figures. It is intentionally unexecuted in the repository.

Run prerequisites:
```pip install -r ../requirements.txt
pip install scikit-learn```

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

ROOT = Path('..')  # notebook lives in notebooks/ so repo path adjustments
LOG_CSV = ROOT / 'runs' / 'ca1' / 'logs.csv'
PARTICLES_NPZ = ROOT / 'runs' / 'ca1' / 'particles.npz'
OUTDIR = ROOT / 'runs' / 'ca1' / 'plots'
OUTDIR.mkdir(parents=True, exist_ok=True)

def plot_loss(df, outpath):
    plt.figure(figsize=(6,4))
    plt.plot(df['step'], df['loss'], label='Sinkhorn loss')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title('Training Loss (Sinkhorn)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=300)
    plt.close()

def plot_w1(df, outpath):
    if 'w1' not in df.columns:
        print('No w1 column; skipping')
        return
    plt.figure(figsize=(6,4))
    plt.plot(df['step'], df['w1'], label='W1 (pred vs MC)')
    plt.xlabel('Step')
    plt.ylabel('W1')
    plt.title('Wasserstein-1 Distance vs MC')
    plt.tight_layout()
    plt.savefig(outpath, dpi=300)
    plt.close()

def plot_epsilon(df, outpath):
    if 'epsilon' not in df.columns:
        print('No epsilon column; skipping')
        return
    plt.figure(figsize=(6,3))
    plt.plot(df['step'], df['epsilon'], label='epsilon')
    plt.xlabel('Step')
    plt.ylabel('Epsilon')
    plt.title('Epsilon Annealing Schedule')
    plt.tight_layout()
    plt.savefig(outpath, dpi=300)
    plt.close()

def plot_particles_scatter(particles, outpath, action=0):
    # particles may be (steps,B,A,N,D) or (B,A,N,D) or (N,D)
    if particles.ndim == 5:
        arr = particles[-1,0,action]
    elif particles.ndim == 4:
        arr = particles[0,action]
    elif particles.ndim == 2:
        arr = particles
    else:
        raise ValueError('Unexpected particles shape: '+str(particles.shape))
    N,D = arr.shape
    plt.figure(figsize=(5,5))
    if D == 1:
        sns.kdeplot(arr.squeeze(), fill=True)
        plt.xlabel('Return')
        plt.title(f'Particle KDE (action={action})')
    elif D == 2:
        plt.scatter(arr[:,0], arr[:,1], s=10, alpha=0.7)
        plt.xlabel('dim0'); plt.ylabel('dim1')
        plt.title(f'Particle scatter (action={action})')
    else:
        from sklearn.decomposition import PCA
        pca = PCA(n_components=2)
        proj = pca.fit_transform(arr)
        plt.scatter(proj[:,0], proj[:,1], s=10, alpha=0.7)
        plt.title(f'Particle PCA projection (action={action})')
    plt.tight_layout()
    plt.savefig(outpath, dpi=300)
    plt.close()

# Main
if LOG_CSV.exists():
    df = pd.read_csv(LOG_CSV)
    if 'loss' in df.columns:
        plot_loss(df, OUTDIR / 'loss.png')
    plot_w1(df, OUTDIR / 'w1.png')
    plot_epsilon(df, OUTDIR / 'epsilon.png')
else:
    print('Log not found; creating dummy loss curve')
    steps = np.linspace(0,1000,200)
    df = pd.DataFrame({'step':steps, 'loss':np.exp(-steps/400.0)+0.01*np.random.randn(len(steps))})
    plot_loss(df, OUTDIR / 'loss.png')

if PARTICLES_NPZ.exists():
    data = np.load(PARTICLES_NPZ)
    key = None
    for k in ['particles','arr','z']:
        if k in data:
            key=k; break
    if key is None:
        key = list(data.files)[0]
    particles = data[key]
    plot_particles_scatter(particles, OUTDIR / 'particles_action0.png', action=0)
else:
    print('Particles file not found; creating dummy particles')
    mode1 = np.random.normal(loc=-1.0, scale=0.2, size=(50,1))
    mode2 = np.random.normal(loc=2.0, scale=0.4, size=(50,1))
    arr = np.vstack([mode1, mode2])
    plot_particles_scatter(arr, OUTDIR / 'particles_action0.png', action=0)

print('Demo plots saved to', OUTDIR)
